# Linux basics: a hands-on notebook

**Cloud Computing · SGH Warsaw School of Economics · Session 2**

This notebook is the practical companion to the [Session 2 study guide](../study-guide-EN.md). The guide explains *why* Linux runs the cloud. Here you run the commands yourself, see what they print, and change them until you understand them.

Every cell works in a small sandbox folder (`~/sgh-linux-lab`), so nothing you do here touches your real files. The last section deletes the sandbox.

| # | Topic | What you'll do |
|---|-------|----------------|
| 1 | Where am I | identify the user, machine, distribution, kernel and resources |
| 2 | The file system | move around the single `/` tree and look at key directories |
| 3 | Working with files | create, copy, move and delete; see why `>` is dangerous |
| 4 | Reading large files | generate a 5,000-line web server log and read it with `head`, `tail`, `wc` |
| 5 | Pipes | answer real questions about the log with `grep`, `cut`, `sort`, `uniq`, `awk` |
| 6 | Redirection and exit codes | `2>&1`, `/dev/null`, `$?`, `&&` and `\|\|` |
| 7 | Permissions | read `rwx`, use `chmod`, hit *Permission denied* on purpose |
| 8 | Processes | background jobs, `kill`, `nohup` |
| 9 | Networking and the metadata service | interfaces, ports, and the `169.254.169.254` trick |
| 10 | Bash scripting | variables, conditions, loops, functions, arguments |
| 11 | Small automations | `find`, `xargs`, `tar` backups with a date in the name |
| 12 | Exercises and clean-up | |

### Where to run this notebook

You need a Jupyter kernel running **on Linux**. Any of these will do:

- the cloud workstation with Jupyter from sessions 3 and 4 (AWS, Azure or GCP),
- Google Cloud Shell or Azure Cloud Shell (`pip install notebook`, then `jupyter notebook`),
- WSL on Windows, or VS Code connected to a Linux VM over SSH.

Cells starting with `%%bash` are run by Bash, not Python. Each one starts a fresh shell, which is why almost every cell begins with `cd "$LAB"`.

> ⚠️ **Windows without WSL:** `%%bash` will pick up Git Bash or nothing at all, and many commands below (`free`, `ip`, `/proc`) won't exist. Use WSL or a cloud machine.
>
> **macOS** works for most sections, but `free`, `ip` and `/proc` are Linux-only. That difference is itself a lesson: macOS is Unix, not Linux.

### Setup
Run this cell first (and again after restarting the kernel). It creates the sandbox and exports `LAB`, so every `%%bash` cell can see it.

In [ ]:
import os, platform, shutil

LAB = os.path.expanduser("~/sgh-linux-lab")
os.makedirs(LAB, exist_ok=True)
os.environ["LAB"] = LAB

print("Sandbox:", LAB)
print("OS:     ", platform.system(), platform.release())
print("bash:   ", shutil.which("bash") or "NOT FOUND - see the note above")
if platform.system() != "Linux":
    print("\nHeads-up: this is not Linux. Some cells will fail; that's expected.")

---
## 1. Where am I and what am I running on

The first thing to do on any new server. Look at the `uname -r` line: on a cloud VM the kernel name usually ends in `-aws`, `-azure` or `-gcp`. Same Ubuntu, different kernel, built for each provider's hypervisor.

In [ ]:
%%bash
echo "User:      $(whoami)"
echo "Hostname:  $(hostname)"
echo "Kernel:    $(uname -r)"
echo "CPU cores: $(nproc)"
echo "Uptime:    $(uptime -p 2>/dev/null || uptime)"
echo
grep -E '^(PRETTY_NAME|ID|VERSION_ID)=' /etc/os-release

In [ ]:
%%bash
free -h        # memory; -h means "human-readable" (MiB, GiB)
echo
df -h /        # disk space on the root file system

`lscpu` prints much more than you need. Pipes (section 5) let you keep only the lines you care about:

In [ ]:
%%bash
lscpu | grep -E '^(Architecture|Model name|CPU\(s\)|Hypervisor vendor|Virtualization type)'

---
## 2. The file system: one tree, starting at `/`

Linux has no `C:` or `D:`. Everything hangs off a single root, `/`, and disks are *mounted* onto branches of that tree.

In [ ]:
%%bash
cd "$LAB"
pwd            # print working directory
echo
ls /           # top level of the tree

In [ ]:
%%bash
cd /etc && pwd     # configuration lives here
cd ~    && pwd     # your home directory (plain "cd" does the same)
cd ..   && pwd     # one level up

`/proc` is not a real directory on disk. The kernel generates these "files" on the fly every time you read them. Many tools (`free`, `uptime`, `ps`) are just nicely formatted views of `/proc`.

In [ ]:
%%bash
head -n 3 /proc/meminfo
echo
cat /proc/loadavg     # load over the last 1, 5 and 15 minutes

---
## 3. Working with files

Linux has no recycle bin. `rm` means gone. Keep that in mind in this section.

In [ ]:
%%bash
cd "$LAB"
mkdir -p project/data/2026/raw     # -p creates the whole path at once
touch project/notes.txt            # empty file
ls -R project                      # -R = recursive

The difference between `>` and `>>` is one character. One overwrites the file, the other appends to it. Watch:

In [ ]:
%%bash
cd "$LAB/project"
echo "First line"  >  notes.txt
echo "Second line" >> notes.txt
echo "Third line"  >> notes.txt
echo "--- after two appends:"; cat notes.txt

echo "Oops" > notes.txt
echo "--- after a single '>':"; cat notes.txt

In [ ]:
%%bash
cd "$LAB/project"
cp notes.txt copy.txt              # copy
mv copy.txt data/                  # move...
mv data/copy.txt data/renamed.txt  # ...or rename: same command
ls -l data
rm data/renamed.txt                # delete a file
rm -r data/2026                    # delete a directory and everything in it
echo "--- after cleanup:"; ls -R .

---
## 4. Reading large files

Servers produce logs, and logs get big. Let's make one. The cell below generates a realistic web server access log with 5,000 requests (same format as Apache or Nginx). The random seed is fixed, so everyone in class gets the same file and the same answers.

In [ ]:
%%bash
cd "$LAB"
awk 'BEGIN {
  srand(42)
  # normal pages, roughly from most to least popular
  np = split("/ / / /index.html /index.html /static/app.js /static/app.js /static/style.css /static/style.css /api/products /api/products /api/orders /login /reports/q3.pdf", pages, " ")
  # traffic by hour: quiet at night, busy during the working day
  split("2 1 1 1 1 2 4 7 10 12 12 11 9 11 12 11 10 8 6 5 4 3 3 2", load, " ")
  n = 0
  while (n < 5000) {
    sec = int(rand() * 86400); h = int(sec / 3600)
    if (rand() * 12 > load[h + 1]) continue
    n++
    if (rand() < 0.05) {                                    # a bot from outside, probing for admin panels
      ip = "203.0.113." int(rand() * 5 + 10)
      p  = (rand() < 0.6 ? "/wp-login.php" : "/admin"); m = "GET"; c = (rand() < 0.85 ? 404 : 403)
    } else {                                                # a normal user on the internal network
      ip = "10.0." int(rand() * 8) "." int(rand() * 40 + 1)
      p  = pages[int(rand() * np) + 1]
      m  = (p == "/login" || p == "/api/orders") && rand() < 0.5 ? "POST" : "GET"
      r  = rand(); c = (r < 0.015 ? 500 : r < 0.04 ? 404 : r < 0.15 ? 304 : 200)
    }
    printf "%s - - [26/Nov/2026:%02d:%02d:%02d +0100] \"%s %s HTTP/1.1\" %s %d\n",
           ip, h, int(sec % 3600 / 60), sec % 60, m, p, c, (c == 304 ? 0 : int(rand() * 50000 + 200))
  }
}' | sort -k4,4 > access.log                                # sort by timestamp, like a real log

ls -lh access.log

In [ ]:
%%bash
cd "$LAB"
wc -l access.log              # how many lines
echo "--- first 3:"
head -n 3 access.log
echo "--- last 3:"
tail -n 3 access.log

Two more you'll use all the time but which don't make sense inside a notebook, because they wait for you:

- `less access.log`: scroll through a file (`/` searches, `q` quits),
- `tail -f /var/log/syslog`: follow a log live as new lines appear (`Ctrl+C` stops).

Try them in a real terminal.

---
## 5. Pipes: the most important idea in this notebook

The `|` symbol passes the output of one program into the next one. Each program does one small thing well. Chain them and you get data analysis without a database.

Let's start with `grep`, which filters lines.

In [ ]:
%%bash
cd "$LAB"
grep -c ' 500 ' access.log            # -c: count matching lines (server errors)
grep -c 'POST' access.log
grep -i 'WP-LOGIN' access.log | head -n 3   # -i: ignore case

**Question 1: which pages are requested most often?**

The request path is the 7th space-separated field. `cut` extracts it, `sort` puts identical values next to each other, `uniq -c` counts them, `sort -rn` orders by count, and `head` keeps the top 5.

If you know SQL, this is `SELECT path, COUNT(*) ... GROUP BY path ORDER BY 2 DESC LIMIT 5`.

In [ ]:
%%bash
cd "$LAB"
cut -d' ' -f7 access.log | sort | uniq -c | sort -rn | head -n 5

Don't take the chain on faith. Remove steps from the end, one at a time, and look at what each stage produces:

In [ ]:
%%bash
cd "$LAB"
echo "--- cut only:";        cut -d' ' -f7 access.log | head -n 4
echo "--- + sort:";          cut -d' ' -f7 access.log | sort | head -n 4
echo "--- + uniq -c:";       cut -d' ' -f7 access.log | sort | uniq -c | head -n 4

**Question 2: who is probing for `/wp-login.php` and `/admin`?**

These pages don't exist on our server. Requests for them come from bots scanning for WordPress and admin panels. A machine with a public IP gets these within minutes of starting up (exercise 2 in the study guide).

In [ ]:
%%bash
cd "$LAB"
grep -E '"(GET|POST) /(wp-login\.php|admin) ' access.log \
  | cut -d' ' -f1 | sort | uniq -c | sort -rn | head -n 5

**Question 3: how many bytes did we send, and what's the error rate?**

`awk` treats each line as columns (`$1`, `$2`, ...) and can do arithmetic. It's a tiny programming language on its own.

In [ ]:
%%bash
cd "$LAB"
awk '{ bytes += $10 } END { printf "Total sent: %.1f MB\n", bytes/1024/1024 }' access.log

awk '{ n++; if ($9 >= 500) err++ } END { printf "5xx errors: %d of %d (%.2f%%)\n", err, n, 100*err/n }' access.log

echo "--- requests per status code:"
awk '{ c[$9]++ } END { for (s in c) print c[s], s }' access.log | sort -rn

**Question 4: what's the busiest hour?**

The timestamp looks like `[26/Nov/2026:14:05:33`. `cut -d: -f2` splits on colons and takes the hour.

In [ ]:
%%bash
cd "$LAB"
cut -d: -f2 access.log | sort | uniq -c | sort -rn | head -n 3

You'll see exactly the same pattern again in session 13 with Hadoop and Spark: *map* (extract a field), *shuffle* (sort), *reduce* (count). There it runs across hundreds of machines. Here it's one line of shell.

---
## 6. Redirection and exit codes

Every program has two output streams: **stdout** (normal output, stream 1) and **stderr** (errors, stream 2). `>` only redirects stdout, so errors still reach the screen.

In [ ]:
%%bash
cd "$LAB"
ls access.log no_such_file > out.txt          # the error still shows up below
echo "--- out.txt contains:"; cat out.txt

ls access.log no_such_file > all.txt 2>&1     # 2>&1: send errors where stdout goes
echo "--- all.txt contains:"; cat all.txt

ls no_such_file 2>/dev/null                   # /dev/null swallows anything
echo "(the error above was silenced)"

Every command also returns an **exit code**: `0` means success, anything else means failure. `$?` holds the code of the last command. Scripts, CI pipelines and Terraform (session 7) all depend on this.

In [ ]:
%%bash
cd "$LAB"
grep -q '/api/orders' access.log; echo "grep found it -> exit code $?"
grep -q '/no/such/page' access.log; echo "grep found nothing -> exit code $?"

# && runs the next command only on success, || only on failure
mkdir -p backup && echo "backup folder is ready"
cd /no/such/dir 2>/dev/null || echo "could not cd, carrying on"

---
## 7. Permissions: who can do what

`ls -l` starts each line with ten characters: the file type (`-` file, `d` directory), then three `rwx` triplets for the **owner**, the **group** and **everyone else**.

In numbers: `r` = 4, `w` = 2, `x` = 1, and you add them up. `640` means owner `rw-`, group `r--`, others `---`.

In [ ]:
%%bash
cd "$LAB"
printf '#!/bin/bash\necho "Hello from a script, running as $(whoami)"\n' > hello.sh
ls -l hello.sh
./hello.sh 2>&1 || echo "^ no execute permission yet (exit code $?)"

In [ ]:
%%bash
cd "$LAB"
chmod u+x hello.sh       # give the owner (u) execute (x) permission
ls -l hello.sh
./hello.sh

In [ ]:
%%bash
cd "$LAB"
touch secret.txt key.pem
chmod 600 secret.txt     # rw------- : only the owner can read and write
chmod 400 key.pem        # r-------- : how SSH private keys must look
ls -l secret.txt key.pem
stat -c '%A  %a  %U:%G  %n' secret.txt key.pem   # same info, also in numeric form

`sudo` runs one command with administrator rights. In the cloud you never log in as `root`. You log in as a normal user and raise privileges only when you need to. It's the principle of least privilege again, which we saw with IAM in session 5.

The commands below need `sudo`, so the cell only **prints** them. Run them yourself in a terminal on your VM.

In [ ]:
%%bash
cat <<'EOF'
sudo apt update                     # refresh the package list (Ubuntu/Debian)
sudo apt install -y htop jq tree    # install a few tools
sudo dnf install -y htop jq tree    # the same on Amazon Linux / RHEL
sudo chown root:root some_file      # change a file's owner
EOF
echo
command -v dpkg >/dev/null && echo "Packages installed on this machine: $(dpkg -l | grep -c '^ii')"

---
## 8. Processes

Every running program is a process with a number (PID). `&` sends a command to the background, `$!` holds the PID of the last background process, and `kill` stops it.

In [ ]:
%%bash
sleep 300 &
PID=$!
echo "Started sleep in the background, PID $PID"
ps -o pid,etime,cmd -p "$PID"
kill "$PID"
sleep 0.5
ps -p "$PID" >/dev/null || echo "Process $PID is gone"

In [ ]:
%%bash
# The 5 processes using the most memory right now
ps aux --sort=-%mem | head -n 6

`nohup` makes a process survive logging out. This is how you start a long R or Python computation on a cloud VM (sessions 3 and 4), close your laptop and come back later for the results.

In [ ]:
%%bash
cd "$LAB"
cat > long_job.sh <<'EOF'
#!/bin/bash
for i in 1 2 3; do
  echo "$(date +%T) step $i of 3"
  sleep 1
done
echo "$(date +%T) done"
EOF
chmod +x long_job.sh

nohup ./long_job.sh > long_job.log 2>&1 &
echo "Job started with PID $!. In real life you could log out now."
sleep 4
cat long_job.log

Services (web servers, SSH, databases) are managed by **systemd**. Containers and WSL often run without it, so the cell checks first.

In [ ]:
%%bash
if command -v systemctl >/dev/null && [ -d /run/systemd/system ]; then
  systemctl list-units --type=service --state=running --no-pager | head -n 10
  echo "..."
  echo "Try: systemctl status ssh   |   journalctl -u ssh --since '1 hour ago'"
else
  echo "systemd is not running here (container or WSL without systemd)."
  echo "On a cloud VM try: systemctl status ssh"
fi

---
## 9. Networking and the metadata service

`ip a` shows the machine's network interfaces. On a cloud VM you'll see a **private** address (e.g. `10.x.x.x`). The public IP you log in through isn't configured on the machine at all. The provider's network translates it.

In [ ]:
%%bash
if command -v ip >/dev/null; then
  ip -brief address
else
  hostname -I 2>/dev/null || echo "'ip' is not installed here"
fi
echo
echo "--- ports listening for connections:"
(ss -tuln 2>/dev/null || netstat -tuln 2>/dev/null || echo "ss/netstat not available") | head -n 10

In [ ]:
%%bash
curl -sI --max-time 5 https://www.sgh.waw.pl | head -n 5     # -I: headers only

So how does a VM find out what it is and where it runs? It asks **`169.254.169.254`**, the metadata service. AWS, Azure and Google Cloud all use the same address, each with its own header.

The cell tries all three with a 1-second timeout. On your laptop all of them fail, and that's the correct answer. On a cloud VM exactly one responds.

In [ ]:
%%bash
M=169.254.169.254
T="--silent --fail --max-time 1"

if r=$(curl $T -H "Metadata-Flavor: Google" "http://$M/computeMetadata/v1/instance/machine-type"); then
  echo "Google Cloud, machine type: ${r##*/}"
elif r=$(curl $T -H "Metadata: true" "http://$M/metadata/instance/compute/vmSize?api-version=2021-02-01&format=text"); then
  echo "Microsoft Azure, VM size: $r"
elif TOKEN=$(curl $T -X PUT "http://$M/latest/api/token" -H "X-aws-ec2-metadata-token-ttl-seconds: 60") && \
     r=$(curl $T -H "X-aws-ec2-metadata-token: $TOKEN" "http://$M/latest/meta-data/instance-type"); then
  echo "AWS, instance type: $r"
else
  echo "No metadata service answered: this is not a cloud VM (or the network blocks it)."
fi

The headers (`Metadata-Flavor`, `Metadata: true`, the AWS token) are there for security. Through this address a VM can also obtain **temporary credentials** for other cloud services. Several well-known data breaches started with an attacker tricking a web application into calling this address on their behalf (an SSRF attack).

---
## 10. Bash scripting

Anything you type by hand can be saved in a file and run again. We'll build up the pieces first, then put them together.

**Variables and command substitution.** No spaces around `=`. `$(...)` captures a command's output.

In [ ]:
%%bash
cd "$LAB"
NAME="SGH"
LINES=$(wc -l < access.log)
TODAY=$(date +%Y-%m-%d)
echo "Hello, $NAME. Today is $TODAY and the log has $LINES lines."
echo 'Single quotes do not expand: $NAME'

**Conditions.** `[ ... ]` is a test. `-f` checks that a file exists, `-gt` means "greater than" for numbers.

In [ ]:
%%bash
cd "$LAB"
ERRORS=$(grep -c ' 500 ' access.log)

if [ "$ERRORS" -gt 50 ]; then
  echo "ALERT: $ERRORS server errors"
elif [ "$ERRORS" -gt 0 ]; then
  echo "Warning: $ERRORS server errors"
else
  echo "All good"
fi

for f in access.log missing.log; do
  if [ -f "$f" ]; then echo "$f exists"; else echo "$f is missing"; fi
done

**Loops.** `for` iterates over a list, and `{1..5}` generates one. Here we split the log into one file per hour, a typical pre-processing step before loading data somewhere.

In [ ]:
%%bash
cd "$LAB"
mkdir -p by_hour
for h in 08 09 10 11 12; do
  grep ":$h:[0-9][0-9]:[0-9][0-9] " access.log > "by_hour/hour_$h.log"
done
wc -l by_hour/*.log

**Functions and arguments.** Inside a script or function, `$1`, `$2`, ... are the arguments, `$#` is how many there are, and `$@` is all of them.

In [ ]:
%%bash
cd "$LAB"
cat > top_paths.sh <<'EOF'
#!/bin/bash
# Usage: ./top_paths.sh LOGFILE [N]
# Prints the N most requested paths (default: 5).

set -euo pipefail          # stop on errors, unset variables and failed pipes

usage() { echo "Usage: $0 LOGFILE [N]" >&2; exit 1; }

[ $# -ge 1 ] || usage
LOG="$1"
N="${2:-5}"                # second argument, or 5 if not given
[ -f "$LOG" ] || { echo "File not found: $LOG" >&2; exit 2; }

echo "Top $N paths in $LOG:"
cut -d' ' -f7 "$LOG" | sort | uniq -c | sort -rn | head -n "$N"
EOF
chmod +x top_paths.sh

./top_paths.sh access.log 3
echo
./top_paths.sh by_hour/hour_10.log
echo
./top_paths.sh nope.log || echo "(exit code $?)"
echo
./top_paths.sh || echo "(exit code $?)"

**Putting it together: the machine report.** This is the `report.sh` script from section 5 of the study guide. The same file, without a single change, works on AWS, Azure, GCP and WSL.

In [ ]:
%%bash
cd "$LAB"
cat > report.sh <<'EOF'
#!/bin/bash
# report.sh - a short report about a cloud machine

echo "=== Report: $(hostname) - $(date '+%Y-%m-%d %H:%M') ==="
echo "System:  $(grep PRETTY_NAME /etc/os-release | cut -d'"' -f2)"
echo "Kernel:  $(uname -r)"
echo "CPU:     $(nproc) cores"
echo "RAM:     $(free -h | awk '/Mem:/ {print $2}')"
echo "Disk /:  $(df -h / | awk 'NR==2 {print $4}') free of $(df -h / | awk 'NR==2 {print $2}')"
echo "Uptime:  $(uptime -p 2>/dev/null || uptime)"

# Identify the provider from the kernel name
case "$(uname -r)" in
  *aws*)       echo "Cloud:   AWS" ;;
  *azure*)     echo "Cloud:   Microsoft Azure" ;;
  *gcp*)       echo "Cloud:   Google Cloud" ;;
  *microsoft*) echo "Cloud:   none - this is WSL on Windows" ;;
  *)           echo "Cloud:   not recognised" ;;
esac
EOF
chmod +x report.sh
./report.sh

---
## 11. Small automations

**`find`** searches by name, size, age and more. **`xargs`** turns a list of files into arguments for another command.

In [ ]:
%%bash
cd "$LAB"
echo "--- all .sh files:"
find . -name '*.sh'
echo "--- files larger than 50 KB:"
find . -type f -size +50k -exec ls -lh {} \;
echo "--- total lines in the hourly logs:"
find by_hour -name '*.log' | xargs cat | wc -l

**A dated backup.** `tar` packs a folder into one compressed archive. Putting the date in the file name means each run creates a new backup instead of overwriting the last one. In practice you'd copy this archive to object storage (session 6): `gsutil cp`, `aws s3 cp` or `az storage blob upload`.

In [ ]:
%%bash
cd "$LAB"
ARCHIVE="backup/logs_$(date +%Y%m%d_%H%M%S).tar.gz"
tar -czf "$ARCHIVE" access.log by_hour/
ls -lh backup/
echo "--- contents:"
tar -tzf "$ARCHIVE"

**Scheduling with cron.** To run a script automatically, for example every night at 2:00, you add a line to your *crontab* (`crontab -e`). The five fields are minute, hour, day of month, month and day of week:

```
# m  h  dom mon dow  command
  0  2  *   *   *    /home/ubuntu/backup.sh >> /home/ubuntu/backup.log 2>&1
*/15 *  *   *   *    /home/ubuntu/report.sh  >> /home/ubuntu/report.log 2>&1
```

The second line runs every 15 minutes. [crontab.guru](https://crontab.guru) translates any schedule into plain English.

In [ ]:
%%bash
crontab -l 2>/dev/null || echo "No crontab for $(whoami) (or cron isn't installed here)."

---
## 12. Exercises

Add a new cell under each question and solve it with a single pipeline or a short script.

1. **Status codes.** How many requests ended with `404`? Which path produced the most of them?
2. **Outsiders.** Our internal network is `10.0.x.x`. List every IP address *outside* that range and how many requests each one made. Hint: `grep -v` inverts a match.
3. **Biggest responses.** Print the 5 requests with the largest response size (field 10). Hint: `sort -k10 -n`.
4. **Traffic per hour.** Using `awk`, print the total number of megabytes sent in each hour, sorted by hour.
5. **Your own script.** Write `errors.sh LOGFILE` that prints the number of `4xx` and `5xx` responses and exits with code `1` if there's more than 1% of `5xx`. Test it on `access.log` and on one of the hourly files.
6. **In the cloud.** Run `report.sh` and the metadata cell from section 9 on a free VM in at least two clouds. What differs?

---
## Clean-up

Removes the sandbox and everything created in it.

In [ ]:
%%bash
rm -rf "$LAB" && echo "Removed $LAB"